In [74]:
# Creating a project following the best practices
# 1. Data Cleaning and Preprocessing
# 2. Train Test Split
# 3. BOW, TF-IDF, Word2Vec
# 4. Training a model using ML

In [75]:
import pandas as pd
df = pd.read_csv("kindle_dataset.csv")

In [76]:
df.head()

,Unnamed: 0.1,Unnamed: 0,asin,helpful,rating,reviewText,reviewTime,reviewerID,reviewerName,summary,unixReviewTime
0,0,11539,B0033UV8HI,"[8, 10]",3,"Jace Rankin may be short, but he's nothing to ...","09 2, 2010",A3HHXRELK8BHQG,Ridley,Entertaining But Average,1283385600
1,1,5957,B002HJV4DE,"[1, 1]",5,Great short read. I didn't want to put it dow...,"10 8, 2013",A2RGNZ0TRF578I,Holly Butler,Terrific menage scenes!,1381190400
2,2,9146,B002ZG96I4,"[0, 0]",3,I'll start by saying this is the first of four...,"04 11, 2014",A3S0H2HV6U1I7F,Merissa,Snapdragon Alley,1397174400
3,3,7038,B002QHWOEU,"[1, 3]",3,Aggie is Angela Lansbury who carries pocketboo...,"07 5, 2014",AC4OQW3GZ919J,Cleargrace,very light murder cozy,1404518400
4,4,1776,B001A06VJ8,"[0, 1]",4,I did not expect this type of book to be in li...,"12 31, 2012",A3C9V987IQHOQD,Rjostler,Book,1356912000


In [77]:
# using only review and ratings
df = df[['reviewText', 'rating']]
df.head()

,reviewText,rating
0,"Jace Rankin may be short, but he's nothing to ...",3
1,Great short read. I didn't want to put it dow...,5
2,I'll start by saying this is the first of four...,3
3,Aggie is Angela Lansbury who carries pocketboo...,3
4,I did not expect this type of book to be in li...,4


In [78]:
df.shape

(12000, 2)

In [79]:
df.isnull().sum()

reviewText    0
rating        0
dtype: int64

In [80]:
df['rating'].unique()

array([3, 5, 4, 2, 1], dtype=int64)

In [81]:
df['rating'].value_counts()

rating
5    3000
4    3000
3    2000
2    2000
1    2000
Name: count, dtype: int64

In [82]:
df['rating']=df['rating'].apply(lambda x: 0 if x<3 else 1) # Coverting ratings to positive(1) and negative(0) reviews

In [83]:
df.head()

,reviewText,rating
0,"Jace Rankin may be short, but he's nothing to ...",1
1,Great short read. I didn't want to put it dow...,1
2,I'll start by saying this is the first of four...,1
3,Aggie is Angela Lansbury who carries pocketboo...,1
4,I did not expect this type of book to be in li...,1


In [84]:
df['rating'].value_counts()

rating
1    8000
0    4000
Name: count, dtype: int64

In [85]:
df['reviewText'] = df['reviewText'].str.lower()

In [86]:
df['reviewText'].head()

0    jace rankin may be short, but he's nothing to ...
1    great short read.  i didn't want to put it dow...
2    i'll start by saying this is the first of four...
3    aggie is angela lansbury who carries pocketboo...
4    i did not expect this type of book to be in li...
Name: reviewText, dtype: str

In [87]:
# Data Cleaning - 1. Removing Special Characters
import re
df['reviewText']= df['reviewText'].apply(lambda x: re.sub('[^a-z A-Z 0-9-]+', "", x))
# There is space between a-z A-Z so that we do not remove the entire spaces from the reviewText columns

In [88]:
df.reviewText.head()

0    jace rankin may be short but hes nothing to me...
1    great short read  i didnt want to put it down ...
2    ill start by saying this is the first of four ...
3    aggie is angela lansbury who carries pocketboo...
4    i did not expect this type of book to be in li...
Name: reviewText, dtype: str

In [89]:
import nltk
nltk.download("stopwords")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\anand\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [90]:
# 2. Removing stopwords
from nltk.corpus import stopwords
df['reviewText'] = df['reviewText'].apply(lambda x: " ".join([y for y in x.split() if y not in stopwords.words("english")]))

In [91]:
# 3. Removing URLs
df['reviewText'] = df['reviewText'].apply(lambda x: re.sub(r"(http|https|ftp|ssh)://([\w_-]+(?:(?:\.[\w_-]+)+))([\w.,@?^=%&:/~+#-]*[\w@?^=%&/~+#-])?", "", str(x)))

In [92]:
# 4. Removing HTML elements
from bs4 import BeautifulSoup
df['reviewText'] = df['reviewText'].apply(lambda x: BeautifulSoup(x, "lxml").get_text())

In [93]:
# 5. Removing additional spaces
df['reviewText'] = df['reviewText'].apply(lambda x: " ".join(x.split()))

In [94]:
df.head()

,reviewText,rating
0,jace rankin may short hes nothing mess man hau...,1
1,great short read didnt want put read one sitti...,1
2,ill start saying first four books wasnt expect...,1
3,aggie angela lansbury carries pocketbooks inst...,1
4,expect type book library pleased find price right,1


In [95]:
# Lemmatization
from nltk.stem import WordNetLemmatizer
wnl = WordNetLemmatizer()

In [96]:
def lemmatize_words(text):
  return " ".join([wnl.lemmatize(word) for word in text.split()])

In [97]:
df['reviewText']= df['reviewText'].apply(lambda x: lemmatize_words(x))

In [98]:
df.head()

,reviewText,rating
0,jace rankin may short he nothing mess man haul...,1
1,great short read didnt want put read one sitti...,1
2,ill start saying first four book wasnt expecti...,1
3,aggie angela lansbury carry pocketbook instead...,1
4,expect type book library pleased find price right,1


In [99]:
# Train test split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(df['reviewText'], df['rating'], test_size=0.2, random_state=42)

In [100]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((9600,), (2400,), (9600,), (2400,))